In [1]:
import os, sys, shutil
from pathlib import Path
try:
    from google.colab import userdata, drive
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
if not os.path.exists('/content/mmai'):
    !git clone https://github.com/michaelyserrano/mmai.git /content/mmai
    !pip install -q -r /content/mmai/project/requirements.txt
%cd /content/mmai
sys.path.insert(0, 'project')
drive.mount('/content/drive')
DRIVE_TB = Path('/content/drive/MyDrive/Colab Notebooks/toolbench_data')
LOCAL_TB = Path('/content/toolbench_data')
if not LOCAL_TB.exists():
    print('Copying toolbench to local disk (first time, slow)...')
    !cp -r "{DRIVE_TB}" "{LOCAL_TB}"
else:
    for fname in ['toolllama_G123_dfs_eval.json', 'toolllama_G123_dfs_train.json']:
        if not (LOCAL_TB / fname).exists():
            shutil.copy(DRIVE_TB / fname, LOCAL_TB / fname)
            print(f'Copied missing {fname}')
TOOLBENCH_DIR = LOCAL_TB

/content/mmai
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Restore pre-computed outputs from Drive if not already on local disk
import shutil
from pathlib import Path
drive_out = Path('/content/drive/MyDrive/Colab Notebooks/mmai/project')
for fname in ['corpus_embeddings.npy', 'api_names.json']:
    dst = Path('project') / fname
    src = drive_out / fname
    if not dst.exists() and src.exists():
        shutil.copy(src, dst)
        print(f'Restored {fname} from Drive')
    elif dst.exists():
        print(f'{fname} already on local disk')
    else:
        print(f'WARNING: {fname} not found — run notebook 01 first')

corpus_embeddings.npy already on local disk
api_names.json already on local disk


In [4]:
import json, numpy as np
from data.load_toolbench import load_api_corpus, load_eval_examples
from data.negative_mining import build_api_lookup, build_random_negatives, build_category_sibling_negatives, build_dfsdt_negatives
from models.embeddings import get_embeddings
from retrieval.retriever import build_faiss_index, retrieve_top_k
from evaluation.metrics import recall_at_k, mean_reciprocal_rank

corpus = load_api_corpus(Path(TOOLBENCH_DIR) / "toolenv" / "tools")
lookup = build_api_lookup(corpus)
corpus_matrix = np.load("project/corpus_embeddings.npy")
with open("project/api_names.json") as f:
    api_names = json.load(f)
name_to_idx = {name: i for i, name in enumerate(api_names)}

evals = load_eval_examples(TOOLBENCH_DIR / "toolllama_G123_dfs_eval.json")
with open(TOOLBENCH_DIR / "toolllama_G123_dfs_eval.json") as f:
    raw_evals = json.load(f)

query_embeddings = get_embeddings([e["user_query"] for e in evals])
print(f"\u2713 Loaded {len(evals)} eval examples")

NameError: name 'json' is not defined

In [ ]:
def evaluate_with_negatives(evals, raw_evals, query_embeddings, corpus_matrix,
                             corpus, lookup, name_to_idx,
                             negative_type: str, n_negatives: int = 99, ks=[1, 5, 10]):
    """
    For each eval example:
      1. Build a restricted candidate pool: ground truth + n negatives
      2. Extract sub-matrix of embeddings for that pool
      3. Build a small FAISS index over just those candidates
      4. Retrieve top-k and evaluate
    """
    recall_scores = {k: [] for k in ks}
    mrr_scores = []

    for i, ex in enumerate(evals):
        raw_ex = raw_evals[ex["raw_idx"]]
        gt_names = ex["ground_truth_apis"]
        gt_indices = [name_to_idx[n] for n in gt_names if n in name_to_idx]
        if not gt_indices:
            continue

        # Build negative pool
        if negative_type == "random":
            negatives = build_random_negatives(corpus, gt_names, n=n_negatives)
        elif negative_type == "sibling":
            negatives = build_category_sibling_negatives(corpus, gt_names, lookup, n=n_negatives)
        elif negative_type == "dfsdt":
            negatives = build_dfsdt_negatives(raw_ex, corpus, gt_names, lookup, n=n_negatives)
        else:
            raise ValueError(f"Unknown negative type: {negative_type}")

        # Build candidate pool: positives + negatives
        candidate_names = gt_names + [a["name"] for a in negatives]
        candidate_indices = [name_to_idx[n] for n in candidate_names if n in name_to_idx]
        if len(candidate_indices) < 2:
            continue

        candidate_embs = corpus_matrix[candidate_indices].astype(np.float32)
        local_index = build_faiss_index(candidate_embs)

        # Map global GT indices to local indices
        global_to_local = {g: j for j, g in enumerate(candidate_indices)}
        local_gt = [global_to_local[g] for g in gt_indices if g in global_to_local]
        if not local_gt:
            continue

        top_k = retrieve_top_k(query_embeddings[i], local_index, k=min(10, len(candidate_indices)))
        for k in ks:
            recall_scores[k].append(recall_at_k(top_k, local_gt, k))
        mrr_scores.append(mean_reciprocal_rank(top_k, local_gt))

    return {
        **{f"recall@{k}": float(np.mean(recall_scores[k])) for k in ks},
        "mrr": float(np.mean(mrr_scores)),
        "n_examples": len(mrr_scores),
    }

In [ ]:
print("Running: Random negatives...")
random_results = evaluate_with_negatives(evals, raw_evals, query_embeddings, corpus_matrix, corpus, lookup, name_to_idx, "random")

print("Running: Category sibling negatives...")
sibling_results = evaluate_with_negatives(evals, raw_evals, query_embeddings, corpus_matrix, corpus, lookup, name_to_idx, "sibling")

print("Running: DFSDT failure-path negatives...")
dfsdt_results = evaluate_with_negatives(evals, raw_evals, query_embeddings, corpus_matrix, corpus, lookup, name_to_idx, "dfsdt")

print("\n=== Hard-Negative Ablation Results ===")
print(f"{'Condition':<25} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'MRR':>8}")
print("-" * 57)
for label, res in [("Random", random_results), ("Category Siblings", sibling_results), ("DFSDT Failure Paths", dfsdt_results)]:
    print(f"{label:<25} {res['recall@1']:>8.4f} {res['recall@5']:>8.4f} {res['recall@10']:>8.4f} {res['mrr']:>8.4f}")

In [ ]:
import matplotlib.pyplot as plt

all_results = {
    "random": random_results,
    "sibling": sibling_results,
    "dfsdt": dfsdt_results,
}
with open("project/results_hard_negatives.json", "w") as f:
    json.dump(all_results, f, indent=2)

# Bar chart: Recall@5 across conditions
labels = ["Random", "Category\nSiblings", "DFSDT\nFailure Paths"]
r_at_5 = [random_results["recall@5"], sibling_results["recall@5"], dfsdt_results["recall@5"]]
colors = ["steelblue", "darkorange", "firebrick"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, r_at_5, color=colors, edgecolor="white")
ax.bar_label(bars, fmt="{:.3f}", padding=3)
ax.set_ylabel("Recall@5")
ax.set_title("Retrieval Difficulty by Negative Pool Type\n(OpenAI text-embedding-3-small baseline)")
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig("project/recall_at_5_ablation.png", dpi=150)
plt.show()
print("\u2713 Saved results and figure")

In [ ]:
import json, shutil
from pathlib import Path
all_results = {'random': random_results, 'sibling': sibling_results, 'dfsdt': dfsdt_results}
with open('project/results_hard_negatives.json', 'w') as f:
    json.dump(all_results, f, indent=2)
drive_out = Path('/content/drive/MyDrive/Colab Notebooks/mmai/project')
drive_out.mkdir(parents=True, exist_ok=True)
shutil.copy('project/results_hard_negatives.json', drive_out / 'results_hard_negatives.json')
print(f'Saved to Drive: {drive_out}')